# Sit 2 — 06. Fix captions v3 (anti-leakage)

**Problema:** captions originales con valores numéricos exactos S5P → −25% PDF.

**Acción:** reescribir `tiles-v3-so2-p99-2021-2024/tiles_meta.parquet`:
- 6 plantillas/clase + 5 modificadores semánticos/clase + slots zona/periodo/NDVI/NDBI.
- Combinatoria efectiva ~600 captions únicas por clase posibles.
- SEED=42 determinístico, backup `.bak` automático.

In [ ]:
import pandas as pd, numpy as np, random, re, shutil
from pathlib import Path

SEED = 42
random.seed(SEED); np.random.seed(SEED)

PARQUET = Path('/home/yeigen/Documents/proyecto-3/tiles/data/tiles-v3-so2-p99-2021-2024/tiles_meta.parquet')
BACKUP = PARQUET.with_suffix('.parquet.bak')

df_src = pd.read_parquet(BACKUP) if BACKUP.exists() else pd.read_parquet(PARQUET)
print(f'cargo de: {"backup" if BACKUP.exists() else "parquet"} | n={len(df_src)}')

In [ ]:
BBOX = (-76.6, 3.3, -76.4, 3.55)
LAT_MED, LON_MED, DELTA = (BBOX[1]+BBOX[3])/2, (BBOX[0]+BBOX[2])/2, 0.02

def zona(lat, lon):
    if lat > LAT_MED+DELTA and lon < LON_MED-DELTA: return 'el noroccidente'
    if lat > LAT_MED+DELTA and lon > LON_MED+DELTA: return 'el nororiente'
    if lat < LAT_MED-DELTA and lon < LON_MED-DELTA: return 'el suroccidente'
    if lat < LAT_MED-DELTA and lon > LON_MED+DELTA: return 'el suroriente'
    if lat > LAT_MED+DELTA: return 'el norte'
    if lat < LAT_MED-DELTA: return 'el sur'
    if lon > LON_MED+DELTA: return 'el oriente'
    if lon < LON_MED-DELTA: return 'el occidente'
    return 'el centro'

def periodo(f):
    m = pd.Timestamp(f).month
    if m in (12,1,2): return 'seco'
    if m in (3,4,5):  return 'lluvioso temprano'
    if m in (6,7,8):  return 'de veranillo'
    return 'lluvioso tardío'

def ndvi_d(v):
    if v >= 0.55: return 'vegetación abundante'
    if v >= 0.35: return 'vegetación moderada'
    if v >= 0.15: return 'vegetación dispersa'
    return 'escasa cobertura vegetal'

def ndbi_d(v):
    if v >= 0.10: return 'tejido urbano denso'
    if v >= 0.00: return 'área parcialmente construida'
    return 'superficie predominantemente natural'

In [ ]:
PLANTILLAS = {
    'contaminacion_alta_NO2': [
        'tile Sentinel-2 con alta columna de dióxido de nitrógeno sobre {zona} de Cali en periodo {periodo}, presenta {ndvi_d} y {ndbi_d}; {modif}',
        'imagen satelital asociada a emisiones elevadas de NO2 en {zona} de Cali con {ndbi_d}; {modif}',
        'paisaje con concentración alta de óxidos de nitrógeno sobre {zona} de Cali en periodo {periodo}, {modif}',
        'observación con NO2 troposférico elevado en {zona} de Cali, {ndvi_d}; {modif}',
        'tile con firma de combustión vehicular sobre {zona} de Cali en periodo {periodo}, {ndbi_d}; {modif}',
        'imagen multiespectral con presencia marcada de NO2 en {zona} de Cali, {ndvi_d} y {ndbi_d}; {modif}',
    ],
    'contaminacion_alta_SO2': [
        'tile Sentinel-2 con alta columna de dióxido de azufre sobre {zona} de Cali en periodo {periodo}, {ndvi_d}; {modif}',
        'imagen asociada a pluma de SO2 sobre {zona} de Cali, {ndbi_d}; {modif}',
        'paisaje con emisión elevada de azufre en {zona} de Cali en periodo {periodo}, {modif}',
        'observación satelital con SO2 vertical alto sobre {zona} de Cali, {ndvi_d}; {modif}',
        'tile compatible con quemas de caña o emisión industrial en {zona} de Cali en periodo {periodo}, {modif}',
        'imagen con firma de SO2 atmosférico sobre {zona} de Cali, {ndbi_d}; {modif}',
    ],
    'ozono_anomalo': [
        'tile Sentinel-2 con ozono troposférico anómalo sobre {zona} de Cali en periodo {periodo}, {modif}',
        'imagen con columna de O3 fuera del rango típico en {zona} de Cali, {ndvi_d}; {modif}',
        'paisaje asociado a actividad fotoquímica regional en {zona} de Cali en periodo {periodo}, {modif}',
        'observación con anomalía de ozono sobre {zona} de Cali, {ndbi_d}; {modif}',
        'tile con ozono troposférico desviado del comportamiento estacional en {zona} de Cali, {modif}',
        'imagen multiespectral con concentración atípica de O3 sobre {zona} de Cali en periodo {periodo}, {modif}',
    ],
    'vegetacion_densa': [
        'tile Sentinel-2 con cobertura vegetal densa sobre {zona} de Cali en periodo {periodo}, {modif}',
        'imagen dominada por canopia continua en {zona} de Cali, {ndbi_d}; {modif}',
        'paisaje con vegetación abundante y baja urbanización sobre {zona} de Cali, {modif}',
        'observación con firma espectral de bosque o cultivo cerrado en {zona} de Cali en periodo {periodo}, {modif}',
        'tile con dosel vegetal continuo sobre {zona} de Cali, {ndbi_d}; {modif}',
        'imagen multiespectral con biomasa vegetal alta en {zona} de Cali en periodo {periodo}, {modif}',
    ],
    'suelo_urbano': [
        'tile Sentinel-2 con tejido urbano denso sobre {zona} de Cali en periodo {periodo}, {modif}',
        'imagen con infraestructura construida continua en {zona} de Cali, {ndvi_d}; {modif}',
        'paisaje urbano con superficies impermeables sobre {zona} de Cali en periodo {periodo}, {modif}',
        'observación dominada por edificación y vías en {zona} de Cali, {ndvi_d}; {modif}',
        'tile con cobertura urbana extendida sobre {zona} de Cali en periodo {periodo}, {modif}',
        'imagen multiespectral con concreto y techo urbano sobre {zona} de Cali, {modif}',
    ],
}

MODIFICADORES = {
    'contaminacion_alta_NO2': ['compatible con tráfico vehicular intenso','consistente con corredor vial congestionado','sugerente de fuente puntual industrial activa','asociada a centro urbano de alta actividad','con patrón espacial típico de emisión antrópica'],
    'contaminacion_alta_SO2': ['compatible con quema de biomasa cañera','sugerente de emisión industrial regional','consistente con planta termoeléctrica o caldera','asociada a corredor industrial Yumbo-Acopi','con pluma de azufre orientada por viento dominante'],
    'ozono_anomalo': ['consistente con fotoquímica intensa de tarde tropical','sugerente de inversión térmica con acumulación de precursores','asociada a episodio de transporte regional','con perfil estacional desviado de la media histórica','compatible con evento de calidad del aire reportable'],
    'vegetacion_densa': ['compatible con caña de azúcar madura','sugerente de bosque seco tropical conservado','consistente con cultivo cerrado de ciclo largo','asociada a corredor ribereño con vegetación riparia','con dosel continuo y baja perturbación visible'],
    'suelo_urbano': ['compatible con tejido residencial consolidado','sugerente de área comercial o industrial construida','consistente con barrio de alta densidad de edificación','asociada a corredor vial principal con uso urbano','con superficie impermeable continua y techos visibles'],
}

In [ ]:
def fila_seed(idx): return (SEED * 1000003 + int(idx)) & 0xFFFFFFFF

def build(row):
    rng = random.Random(fila_seed(row.name))
    plant = rng.choice(PLANTILLAS[row.clase])
    modif = rng.choice(MODIFICADORES[row.clase])
    return plant.format(
        zona=zona(row.lat, row.lon),
        periodo=periodo(row.fecha_s2),
        ndvi_d=ndvi_d(row.ndvi),
        ndbi_d=ndbi_d(row.ndbi),
        modif=modif,
    )

df = df_src.copy()
df['texto_old'] = df['texto']
df['texto'] = df.apply(build, axis=1)

ur = df.texto.nunique() / len(df)
print(f'unique_caption_ratio = {ur:.3f}')
for c in sorted(df.clase.unique()):
    sub = df[df.clase==c]
    print(f'  {c:30s} | {sub.texto.nunique():3d}/{len(sub)} ({sub.texto.nunique()/len(sub):.2%})')

In [ ]:
NOMBRES = re.compile(r'(?:Sentinel-2|NO2|SO2|O3|S5P|S2|ERA5|MODIS|B[0-9]+)', re.IGNORECASE)
df['_leak'] = df.texto.apply(lambda t: bool(re.search(r'\d', NOMBRES.sub('', t))))

print('=== CHECKLIST ANTI-LEAKAGE ===')
for tok in ['tile_','cell_','hash','MGRS','mol/m2','NDVI=','NDBI=','e-0','e+0']:
    h = df.texto.str.contains(tok, case=False, regex=False).sum()
    print(f'  [{"FAIL" if h else "OK  "}] "{tok}": {h}')
print(f'  [{"FAIL" if df._leak.sum() else "OK  "}] digitos fuera de productos: {df._leak.sum()}')

if not BACKUP.exists():
    shutil.copy(PARQUET, BACKUP)
    print(f'\nbackup creado: {BACKUP}')
df_out = df.drop(columns=['texto_old','_leak'])
df_out.to_parquet(PARQUET, index=False)
print(f'parquet OK: {PARQUET}')

print('\n=== 2 captions por clase ===')
for c in sorted(df_out.clase.unique()):
    for t in df_out[df_out.clase==c].texto.sample(2, random_state=SEED).tolist():
        print(f'[{c}] {t}')